In [1]:
import numpy as np

convert = np.array([[1, 0, 0, 0],
                    [0, -1, 0, 0],
                    [0, 0, -1, 0],
                    [0, 0, 0, 1]]).astype(np.float32)

In [2]:
inv_convert = np.linalg.inv(convert)

In [4]:
json_str = """
{'data_source': 'Web',
 'camera': [-0.9413449764251709,
  -0.13336405158042908,
  0.3099737763404846,
  -0.8369292616844177,
  -0.002956461627036333,
  -0.9152934551239014,
  -0.40277689695358276,
  1.087497591972351,
  0.3374329209327698,
  -0.38006842136383057,
  0.8612124919891357,
  -2.3252737522125244,
  0.0,
  0.0,
  0.0,
  1.0,
  4.2647,
  0.0,
  0.5,
  0.0,
  4.2647,
  0.5,
  0.0,
  0.0,
  1.0],
 'hpose': [-160.27944946289062, 0.1799468994140625, -23.812721252441406],
 'align_box': [124.04750593824228,
  141.07363420427555,
  760.0950118764846,
  760.0950118764846],
 'align_quad': [[124.61042695034145, 141.29198058937055],
  [124.61042695034145, 900.8631235154395],
  [884.1815617577198, 900.8631235154395],
  [884.1815617577198, 141.29198058937055]],
 'view': 'back',
 'head_image_path': 'head_images/008293_00.png',
 'head_parsing_path': 'head_parsing/008293_00.png',
 'align_image_path': 'align_images/008293_00.png',
 'align_parsing_path': 'align_parsing/008293_00.png',
 'camera_scoord': [-70.20481650648958,
  66.24810915319013,
  2.699999930432925,
  -0.8369292616844177,
  1.087497591972351,
  -2.3252737522125244],
 'svd_score': 0.5838066040729634,
 'laplacian_score': 641.7263444411291,
 'iv2b_ratio': 0.2888778555751773,
 'head_region_thresh': 0.5518110603876089,
 'dup_num': 2}
"""
import json
data = json.loads(json_str.replace("'", '"'))

In [8]:
from scipy.spatial.transform import Rotation

def R2hpose(R_matrix):
    """degree"""
    r_hpose = Rotation.from_matrix(R_matrix).as_euler('zyx', degrees=True)
    r_roll, r_yaw, r_pitch = r_hpose.astype(np.float32).tolist()
    pitch, yaw, roll = -r_pitch, -r_yaw, -r_roll
    hpose = [yaw, roll, pitch]
    return hpose


In [10]:
# cam2world
cam2world = np.array(data["camera"][:16]).reshape(4, 4)
P = np.linalg.inv(cam2world)
R_in = inv_convert @ P
R_in = R_in[:3, :3]
hpose = R2hpose(R_in)

In [11]:
hpose

[-19.720550537109375, -179.82005310058594, 156.18727111816406]

In [12]:
data["hpose"]

[-160.27944946289062, 0.1799468994140625, -23.812721252441406]

[-19.720550537109375, -179.82005310058594, 156.18728637695312]

In [44]:
def hpose2R(hpose):
    # print(hpose)
    yaw, roll, pitch = hpose
    # print(yaw, roll, pitch)
    r_pitch, r_yaw, r_roll = -pitch, -yaw, -roll
    # print(r_pitch, r_yaw, r_roll)
    R = Rotation.from_euler('zyx', [r_roll, r_yaw, r_pitch], degrees=True).as_matrix()
    return R

def R2hpose(R_matrix):
    """degree"""
    r_hpose = Rotation.from_matrix(R_matrix).as_euler('zyx', degrees=True)
    # print(r_hpose)
    r_roll, r_yaw, r_pitch = r_hpose.astype(np.float32).tolist()
    # print(r_roll, r_yaw, r_pitch)
    pitch, yaw, roll = -r_pitch, -r_yaw, -r_roll
    # print(pitch, yaw, roll)
    hpose = [yaw, roll, pitch]
    # print(hpose)
    return hpose

In [49]:
hpose2R(data["hpose"])

array([[-0.94134493, -0.00295646,  0.33743292],
       [ 0.13336404,  0.91529341,  0.38006842],
       [-0.30997378,  0.4027769 , -0.86121253]])

In [50]:
np.array(data['camera'])[:16].reshape(4, 4)[:3, :3]

array([[-0.94134498, -0.13336405,  0.30997378],
       [-0.00295646, -0.91529346, -0.4027769 ],
       [ 0.33743292, -0.38006842,  0.86121249]])

In [51]:
hpose2R(R2hpose(hpose2R(data["hpose"])))

array([[-0.94134493, -0.00295646,  0.33743292],
       [ 0.133364  ,  0.91529347,  0.3800683 ],
       [-0.3099738 ,  0.40277677, -0.86121258]])

In [57]:
hpose1 = data["hpose"]
print(f"hpose1: {hpose1}")
hpose2 = R2hpose(hpose2R(data["hpose"]))
print(f"hpose2: {hpose2}")

R1 = hpose2R(hpose1)
print(f"R1: \n{R1}")
R2 = hpose2R(hpose2)
print(f"R2: \n{R2}")
print(f"R1-R2: {np.abs(R1 - R2).sum()}")


hpose1: [-160.27944946289062, 0.1799468994140625, -23.812721252441406]
hpose2: [-19.720550537109375, -179.82005310058594, 156.18728637695312]
R1: 
[[-0.94134493 -0.00295646  0.33743292]
 [ 0.13336404  0.91529341  0.38006842]
 [-0.30997378  0.4027769  -0.86121253]]
R2: 
[[-0.94134493 -0.00295646  0.33743292]
 [ 0.133364    0.91529347  0.3800683 ]
 [-0.3099738   0.40277677 -0.86121258]]
R1-R2: 3.998322256060227e-07


In [53]:
data["hpose"]

[-160.27944946289062, 0.1799468994140625, -23.812721252441406]

In [52]:
R2hpose(hpose2R(data["hpose"]))

[-19.720550537109375, -179.82005310058594, 156.18728637695312]

In [54]:
R2hpose(hpose2R(R2hpose(hpose2R(data["hpose"]))))

[-19.720550537109375, -179.82005310058594, 156.18728637695312]

In [58]:
np.abs(R2hpose(hpose2R(data["hpose"]))) + np.abs(data["hpose"])

array([180.        , 180.        , 180.00000763])

In [ ]:
# print(data["hpose"])

[-160.27944946289062, 0.1799468994140625, -23.812721252441406]
-160.27944946289062 0.1799468994140625 -23.812721252441406
23.812721252441406 160.27944946289062 -0.1799468994140625
[ 179.8200531    19.72055054 -156.18727875]
179.82005310058594 19.720550537109375 -156.18728637695312
156.18728637695312 -19.720550537109375 -179.82005310058594
[-19.720550537109375, -179.82005310058594, 156.18728637695312]
[-19.720550537109375, -179.82005310058594, 156.18728637695312]


In [32]:
yaw, roll, pitch = data["hpose"]
R = Rotation.from_euler('zyx', [yaw, roll, pitch], degrees=True).as_matrix()

In [33]:
yaw, roll, pitch

(-160.27944946289062, 0.1799468994140625, -23.812721252441406)

In [34]:
Rotation.from_matrix(R).as_euler('zyx', degrees=True)

array([-160.27944946,    0.1799469 ,  -23.81272125])

In [ ]:



def eg3dcamparams(R_in):
    # World Coordinate System: x(right), y(up), z(forward)
    camera_dist = 2.7
    intrinsics = np.array([[4.2647, 0, 0.5], [0, 4.2647, 0.5], [0, 0, 1]])
    # assume inputs are rotation matrices for world2cam projection
    R = np.array(R_in).astype(np.float32).reshape(4,4)
    # add camera translation
    t = np.eye(4, dtype=np.float32)
    t[2, 3] = - camera_dist

    # convert to OpenCV camera
    convert = np.array([
        [1, 0, 0, 0],
        [0, -1, 0, 0],
        [0, 0, -1, 0],
        [0, 0, 0, 1],
    ]).astype(np.float32)

    # world2cam -> cam2world
    P = convert @ t @ R
    cam2world = np.linalg.inv(P)

    # add intrinsics
    label_new = np.concatenate([cam2world.reshape(16), intrinsics.reshape(9)], -1)
    return label_new


def hpose2camera(hpose):
    R = hpose2R(hpose)
    s = 1
    t3d = np.array([0., 0., 0.])
    R[:, :3] = R[:, :3] * s
    P = np.concatenate([R, t3d[:, None]], 1)
    P = np.concatenate([P, np.array([[0, 0, 0, 1.]])], 0)
    cam = eg3dcamparams(P.flatten())
    return cam

In [18]:
hpose2camera(data["hpose"])

t [[ 1.   0.   0.   0. ]
 [ 0.   1.   0.   0. ]
 [ 0.   0.   1.  -2.7]
 [ 0.   0.   0.   1. ]]


array([-9.41344976e-01, -1.33364052e-01,  3.09973776e-01, -8.36929262e-01,
       -2.95646163e-03, -9.15293455e-01, -4.02776897e-01,  1.08749759e+00,
        3.37432921e-01, -3.80068421e-01,  8.61212492e-01, -2.32527375e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  1.00000000e+00,
        4.26470000e+00,  0.00000000e+00,  5.00000000e-01,  0.00000000e+00,
        4.26470000e+00,  5.00000000e-01,  0.00000000e+00,  0.00000000e+00,
        1.00000000e+00])

array([-9.41344976e-01, -1.33364052e-01,  3.09973776e-01, -8.36929262e-01,
       -2.95646163e-03, -9.15293455e-01, -4.02776897e-01,  1.08749759e+00,
        3.37432921e-01, -3.80068421e-01,  8.61212492e-01, -2.32527375e+00,
        0.00000000e+00,  0.00000000e+00,  0.00000000e+00,  1.00000000e+00,
        4.26470000e+00,  0.00000000e+00,  5.00000000e-01,  0.00000000e+00,
        4.26470000e+00,  5.00000000e-01,  0.00000000e+00,  0.00000000e+00,
        1.00000000e+00])

In [19]:
cam2world = np.array(data["camera"])[:16].reshape(4, 4)
P = np.linalg.inv(cam2world)

camera_dist = 2.7
t = np.eye(4, dtype=np.float32)
t[2, 3] = - camera_dist

# convert to OpenCV camera
convert = np.array([
    [1, 0, 0, 0],
    [0, -1, 0, 0],
    [0, 0, -1, 0],
    [0, 0, 0, 1],
]).astype(np.float32)

R_in = np.linalg.inv(t) @ np.linalg.inv(convert) @ P

In [21]:
R = R_in[:3, :3]

In [22]:
hpose = R2hpose(R)

In [23]:
hpose

[-19.720550537109375, -179.82005310058594, 156.18727111816406]

In [24]:
data['hpose']

[-160.27944946289062, 0.1799468994140625, -23.812721252441406]